# TF-IDF 트랙 전처리

**담당**: 영민  
**데이터**: AI Hub 낚시성 기사 탐지 데이터 1세부 (`work_pool`)  
**목표**: TF-IDF 벡터화를 위한 전처리 + 유형별 키워드 분석

---

## 전체 단계
1. **데이터 로딩** ✅
2. **텍스트 정제** ✅ (이전 단계 완료, parquet에 저장됨)
3. **형태소 분석 (Komoran)** ✅ (이전 단계 완료, parquet에 저장됨)
4. **불용어 제거 + 구두점 신호 추가** ← 수정됨
5. **TF-IDF 벡터화**
6. **키워드 분석 (Class-level TF-IDF)**

---

## ⚠️ youngmin_guide.md 규칙 (수업 p.60 근거)

EDA 결과:
- `?` → 낚시성 기사에서 **3.2배** 높게 나옴
- `...` → 낚시성 기사에서 **4.3배** 높게 나옴

**문제**: Komoran 형태소 분석에서 `?`(SF 태그), `...`(SE 태그)는 NNG/NNP/VV/VA 필터에 걸리지 않아  
`title_morphs`에서 사라진다.

**해결**: `title_tfidf` 컬럼(구두점 보존됨)을 참조하여  
`?` → `HAS_QUESTION`, `...` → `HAS_ELLIPSIS` 특수 토큰을 `title_tokens`에 추가한다.

In [ ]:
import pandas as pd
import numpy as np
import re
import time

df = pd.read_parquet('data/processed/work_pool_tfidf_tokens.parquet')
print(f'불러오기 완료: {len(df):,}건')
print(f'컬럼: {list(df.columns)}')

# title_tfidf 컬럼 존재 확인 (? 와 ... 보존된 컬럼)
assert 'title_tfidf' in df.columns, '❌ title_tfidf 컬럼이 없습니다'
assert 'title_morphs' in df.columns, '❌ title_morphs 컬럼이 없습니다'
print('\n✅ title_tfidf, title_morphs 컬럼 확인 완료')

---

## Step 1-4. 불용어 제거 + 구두점 신호 추가

### 처리 순서
1. `title_morphs`에서 불용어 제거 (기존 로직)
2. `title_tfidf`에 `?` 있으면 → `HAS_QUESTION` 추가
3. `title_tfidf`에 `...` 있으면 → `HAS_ELLIPSIS` 추가

`title_morphs` 자체는 수정하지 않는다 (원본 보존).

In [ ]:
STOPWORDS = {
    # 의존명사
    '것', '수', '들', '등', '때', '곳', '점', '뿐', '만큼', '정도', '경우',
    '상황', '입장', '가운데', '속', '중', '전', '후', '내', '위', '아래',
    '사이', '동안', '이상', '이하', '이후', '이전', '이번', '지난',
    # 매우 일반적인 명사
    '관련', '문제', '사실', '이유', '결과', '방법', '내용', '부분',
    '기자', '뉴스', '기사', '보도', '발표', '공개', '확인',
    '예정', '계획', '방침', '결정', '진행', '실시', '추진',
    '올해', '최근', '지금', '현재', '앞서', '이어', '또한', '함께',
    # 매우 일반적인 동사/형용사
    '있다', '없다', '하다', '되다', '이다', '아니다', '같다', '받다',
    '갖다', '나다', '보다', '오다', '가다', '알다', '말다',
    '크다', '작다', '많다', '적다', '좋다', '나쁘다',
    '통해', '위해', '대해', '따라', '따른', '의한', '위한', '통한',
    # 숫자 단위
    '년', '월', '일', '시', '분', '초', '개', '명', '원', '달러',
    '억', '조', '천', '백', '만',
    # 변별력 없는 어간
    '위하',
}

def remove_stopwords(token_str):
    """title_morphs에서 불용어, 숫자 토큰, 꺽쇠 토큰, 1글자를 제거한다."""
    if not isinstance(token_str, str) or not token_str.strip():
        return ''
    tokens = token_str.split()
    filtered = [
        tok for tok in tokens
        if len(tok) >= 2
        and tok not in STOPWORDS
        and not tok.isdigit()
        and not re.match(r'^<.+>$', tok)   # <NUM> 등 제거
    ]
    return ' '.join(filtered)


def build_title_tokens(row):
    """
    Komoran 형태소 결과(title_morphs)에서 불용어를 제거하고,
    title_tfidf에서 '?' 와 '...' 신호를 특수 토큰으로 추가한다.

    EDA 근거 (youngmin_guide.md, preprocessing_pipeline.md 섹션 9):
      - '?'   낚시성에서 3.2배 높음 → HAS_QUESTION
      - '...' 낚시성에서 4.3배 높음 → HAS_ELLIPSIS

    Komoran은 '?'(SF태그), '...'(SE태그)를 NNG/NNP/VV/VA 필터에서 버리므로
    title_morphs에 이 신호가 없다. title_tfidf(원본 보존)에서 감지하여 추가한다.
    """
    tokens = remove_stopwords(row['title_morphs'])
    original = str(row['title_tfidf'])

    if '?' in original:
        tokens += ' HAS_QUESTION'
    if '...' in original:
        tokens += ' HAS_ELLIPSIS'

    return tokens.strip()


print('함수 정의 완료')

# 동작 확인
test_cases = [
    {'title_morphs': '정부 경제 <NUM> 발표 충격', 'title_tfidf': '정부 경제 발표...충격'},
    {'title_morphs': '코로나 백신 부작용 나오',    'title_tfidf': '코로나 백신 부작용이 나온다고?'},
    {'title_morphs': '미국 중국 갈등 심화',        'title_tfidf': '미국-중국 갈등 심화?...'},
]
print('\n[동작 확인]')
for tc in test_cases:
    result = build_title_tokens(tc)
    print(f'  입력: "{tc["title_tfidf"]}"')
    print(f'  결과: "{result}"')
    print()

In [ ]:
print('전체 데이터 처리 중...')
t = time.time()
df['title_tokens'] = df.apply(build_title_tokens, axis=1)
elapsed = time.time() - t
print(f'완료! ({elapsed:.1f}초)')

before = df['title_morphs'].str.split().str.len().mean()
after  = df['title_tokens'].str.split().str.len().mean()
print(f'평균 토큰 수: {before:.1f} → {after:.1f}개')

# HAS_QUESTION, HAS_ELLIPSIS 추가 확인
n_q = df['title_tokens'].str.contains('HAS_QUESTION').sum()
n_e = df['title_tokens'].str.contains('HAS_ELLIPSIS').sum()
total = len(df)
print(f'\n구두점 신호 통계:')
print(f'  HAS_QUESTION: {n_q:,}건 ({n_q/total*100:.1f}%)')
print(f'  HAS_ELLIPSIS: {n_e:,}건 ({n_e/total*100:.1f}%)')

# 낚시성 vs 정상 비율 확인 (EDA 결과 검증)
cb  = df[df['binary_label'] == 1]
ncb = df[df['binary_label'] == 0]
q_cb_ratio  = cb['title_tokens'].str.contains('HAS_QUESTION').mean()
q_ncb_ratio = ncb['title_tokens'].str.contains('HAS_QUESTION').mean()
e_cb_ratio  = cb['title_tokens'].str.contains('HAS_ELLIPSIS').mean()
e_ncb_ratio = ncb['title_tokens'].str.contains('HAS_ELLIPSIS').mean()
print(f'\nEDA 검증 (낚시성 vs 정상 비율):')
print(f'  HAS_QUESTION — 낚시 {q_cb_ratio:.3f} / 정상 {q_ncb_ratio:.3f} → {q_cb_ratio/q_ncb_ratio:.1f}배 (기대: ~3.2배)')
print(f'  HAS_ELLIPSIS — 낚시 {e_cb_ratio:.3f} / 정상 {e_ncb_ratio:.3f} → {e_cb_ratio/e_ncb_ratio:.1f}배 (기대: ~4.3배)')

# 샘플 출력
print('\n[샘플: ? 포함 제목]')
sample_q = df[df['title_tokens'].str.contains('HAS_QUESTION')].head(3)
for _, row in sample_q.iterrows():
    print(f'  원본: {row["title_tfidf"]}')
    print(f'  토큰: {row["title_tokens"]}')
    print()

print('[샘플: ... 포함 제목]')
sample_e = df[df['title_tokens'].str.contains('HAS_ELLIPSIS')].head(3)
for _, row in sample_e.iterrows():
    print(f'  원본: {row["title_tfidf"]}')
    print(f'  토큰: {row["title_tokens"]}')
    print()

In [ ]:
save_cols = ['newsID', 'binary_label', 'type_label', 'source_class',
             'title_clean', 'title_tfidf', 'title_morphs', 'title_tokens']
df[save_cols].to_parquet('data/processed/work_pool_tfidf_tokens.parquet', index=False)
print('저장 완료 ✅')

---

## Step 1-5. TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df['title_tokens'].fillna('').tolist()

print('TF-IDF 벡터화 시작...')
t = time.time()
# HAS_QUESTION, HAS_ELLIPSIS가 피처로 들어감
vectorizer = TfidfVectorizer(max_features=10_000, min_df=5, max_df=0.95)
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'완료! ({time.time()-t:.1f}초)')
print(f'행렬 크기: {tfidf_matrix.shape[0]:,} × {tfidf_matrix.shape[1]:,}')
print(f'어휘 사전: {len(vectorizer.vocabulary_):,}개')
print(f'행렬 밀도: {tfidf_matrix.nnz/(tfidf_matrix.shape[0]*tfidf_matrix.shape[1])*100:.3f}%')

# HAS_QUESTION, HAS_ELLIPSIS가 어휘에 포함됐는지 확인
vocab = vectorizer.vocabulary_
print(f'\n구두점 피처 확인:')
print(f'  HAS_QUESTION: {"포함 ✅" if "has_question" in vocab else "미포함 ❌"}')
print(f'  HAS_ELLIPSIS: {"포함 ✅" if "has_ellipsis" in vocab else "미포함 ❌"}')

---

## Step 1-6. 키워드 분석 (Class-level TF-IDF)

In [ ]:
def class_tfidf_keywords(df, label_col, label_map, token_col='title_tokens', n=20):
    class_docs  = {}
    class_sizes = {}
    for label_id, label_name in label_map.items():
        mask = df[label_col] == label_id
        class_docs[label_name]  = ' '.join(df.loc[mask, token_col].fillna(''))
        class_sizes[label_name] = int(mask.sum())

    doc_list    = list(class_docs.values())
    class_names = list(class_docs.keys())

    cv   = TfidfVectorizer(max_features=20_000, min_df=1)
    mat  = cv.fit_transform(doc_list)
    feat = cv.get_feature_names_out()

    results = {}
    for i, cname in enumerate(class_names):
        scores  = mat[i].toarray().flatten()
        top_idx = scores.argsort()[::-1][:n]
        results[cname] = [(feat[j], scores[j]) for j in top_idx]
    return results, class_sizes

print('class_tfidf_keywords 함수 정의 완료')

In [ ]:
binary_map = {0: '정상', 1: '낚시성'}
binary_kw, binary_sizes = class_tfidf_keywords(df, 'binary_label', binary_map, n=20)

clickbait_kw = binary_kw['낚시성']
normal_kw    = binary_kw['정상']

print('='*58)
print(f'{"낚시성 기사 특징 키워드 TOP20":^27} | {"정상 기사 특징 키워드 TOP20":^27}')
print('='*58)
for (cw, cs), (nw, ns) in zip(clickbait_kw, normal_kw):
    print(f'{cw:>15} ({cs:.4f})  |  {nw:<15} ({ns:.4f})')

In [ ]:
type_map = {
    0: '의문유발-부호(11)', 1: '의문유발-은닉(12)', 2: '선정표현(13)',
    3: '속어/줄임말(14)',   4: '사실과대(15)',      5: '주어왜곡(16)'
}
direct_df = df[df['type_label'] != -1].copy()
type_kw, type_sizes = class_tfidf_keywords(direct_df, 'type_label', type_map, n=15)

print('=== 낚시성 유형별 특징 키워드 TOP 15 (Class-level TF-IDF) ===')
for type_name, keywords in type_kw.items():
    kw_str = ', '.join([f'{w}({s:.3f})' for w, s in keywords])
    print(f'\n[{type_name}] ({type_sizes[type_name]:,}건)')
    print(f'  {kw_str}')

In [ ]:
import os
print('='*60)
print('TF-IDF 트랙 전처리 완료!')
print('='*60)
fsize = os.path.getsize('data/processed/work_pool_tfidf_tokens.parquet')/1024/1024
print(f'\n저장 파일: data/processed/work_pool_tfidf_tokens.parquet ({fsize:.1f} MB)')
print(f'tfidf_matrix: {tfidf_matrix.shape} 희소행렬')
print(f'vectorizer  : 어휘 {len(vectorizer.vocabulary_):,}개')
print('\n[수정 내역]')
print('  ✅ HAS_QUESTION 토큰: ? 포함 제목 신호 (EDA: 낚시성 3.2배)')
print('  ✅ HAS_ELLIPSIS 토큰: ... 포함 제목 신호 (EDA: 낚시성 4.3배)')
print('\n[다음 단계]')
print('  → tfidf_matrix로 분류 모델 학습 (Logistic Regression, SVM 등)')
print('  → 5-Fold Stratified Cross-Validation')